In [ ]:
import pandas as pd
import numpy as np

# Load your cleaned results table
results = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\results_clean.csv')

print("--- PRE-MERGE AUDIT ---")

# 1. Grain Check: Is it strictly one row per driver, per race?
duplicates = results.duplicated(subset=['raceId', 'driverId'], keep=False)
num_dupes = duplicates.sum()
print(f"1. Grain Check: {num_dupes} duplicate (raceId, driverId) rows found.")
if num_dupes == 0:
    print("   ✅ PASS: Grain is correct.")
else:
    print("   ❌ FAIL: You must drop duplicates before merging.")

# 2. Sentinel Check: Are there any '\N' strings left?
has_N = (results == '\\N').any().any()
if not has_N:
    print("   ✅ PASS: No '\\N' sentinels found.")
else:
    print("   ❌ FAIL: '\\N' strings still exist. Replace with np.nan.")

# 3. Target Creation: Create the podium target now (Pre-merge)
print(f"3. Target 'podium' created. Baseline podium rate: {results['podium'].mean():.1%}")

## Adding driver pre-race points and wins

In [ ]:
import pandas as pd

# 1. Load the necessary cleaned tables
results = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\results_clean.csv')
races = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\races_clean.csv')
driver_standings = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\driver_standings_clean.csv')
qualifying = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\qualifying_clean_with_qs&pos.csv')

# 2. Merge standings with races to get year and round
driver_standings = driver_standings.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')

# 3. Sort chronologically by driver, then year, then round
driver_standings = driver_standings.sort_values(by=['driverId', 'race_year', 'race_round'])

# 4. Safely shift the standings! 
# (FIXED: using 'race_year' instead of 'year')
driver_standings['pre_race_points'] = driver_standings.groupby(['driverId', 'race_year'])['driver_points_cumm'].shift(1).fillna(0)
driver_standings['pre_race_wins'] = driver_standings.groupby(['driverId', 'race_year'])['driver_wins_cumm'].shift(1).fillna(0)

# 5. Calculate 'won_previous_race' BEFORE dropping the race_year column
# (FIXED: moved up here, and using 'race_year')
driver_standings['won_previous_race'] = driver_standings.groupby(['driverId', 'race_year'])['pre_race_wins'].diff().fillna(0)

# 6. Drop the original post-race leakage columns before joining to your main table
# (ADDED 'won_previous_race' so it doesn't get deleted)
driver_standings = driver_standings[['raceId', 'driverId', 'pre_race_points', 'pre_race_wins', 'won_previous_race']]

# 7. Save to CSV
driver_standings.to_csv('..\\ACL_9th-F1\\F1_clean\\driver_standings_clean_pre_race.csv', index=False)

print("Shifted standings successfully saved!")

In [ ]:
driver_standings.head(100)